# Gaussian Mixture Model with EM

**Difficulty**: 🟡 Medium

---

### Problem Statement

The Gaussian mixture model is the canonical latent-variable model, and **expectation-maximization
(EM)** is the canonical way to fit it. GMMs are used for soft clustering, anomaly detection, color
and background segmentation, Fisher-vector image encoding, and codebook initialization. The EM
lower bound is also the same ELBO that VAEs and diffusion models optimize. The model is

$$p(x) = \sum_{k=1}^{K} \pi_k\, \mathcal N(x \mid \mu_k, \Sigma_k), \qquad \pi_k \ge 0,\ \sum_k \pi_k = 1 .$$

EM alternates two closed-form steps that never decrease the likelihood:

- **E-step**: the responsibilities $r_{nk} = p(k \mid x_n) = \dfrac{\pi_k \mathcal N(x_n\mid\mu_k,\Sigma_k)}{\sum_j \pi_j \mathcal N(x_n\mid\mu_j,\Sigma_j)}$.
- **M-step**: with $N_k = \sum_n r_{nk}$,
  $$\pi_k = \frac{N_k}{N},\qquad \mu_k = \frac{1}{N_k}\sum_n r_{nk}\,x_n,\qquad \Sigma_k = \frac{1}{N_k}\sum_n r_{nk}\,(x_n-\mu_k)(x_n-\mu_k)^{\top} + \texttt{reg\_covar}\cdot I .$$

In more than a couple of dimensions the densities underflow to 0 in floating point, so the
E-step has to be done in **log space**.

---

### Requirements

1. **`GaussianMixture(n_components, max_iter=100, tol=1e-4, reg_covar=1e-6, seed=0)`** with full covariances.
2. **`fit(X)`**: `X` is `(N, D)`. Returns `self` and sets
   - `weights_` `(K,)`, `means_` `(K, D)`, `covariances_` `(K, D, D)`;
   - `log_likelihood_history_`: a list of floats. Entry $i$ is the **mean log-likelihood per sample**
     $\frac1N\sum_n \log p(x_n)$ of the parameters **after $i$ M-steps**, so entry 0 is the
     initialization.
3. **Loop and stopping**: run at most `max_iter` M-steps. Stop as soon as an M-step improves the
   mean log-likelihood by **less than `tol`**. The history therefore has between 2 and
   `max_iter + 1` entries, and `log_likelihood_history_[-1] == score(X)` for the fitted model.
4. **E-step in log space**: $\log r_{nk} = \log\pi_k + \log\mathcal N(x_n\mid\mu_k,\Sigma_k) - \operatorname{logsumexp}_j(\cdot)$,
   with `torch.logsumexp`. Responsibilities must stay finite even for points far from every component.
5. **M-step** exactly as above: divide by $N_k$ (not $N$, not $N_k - 1$), use the *new* means in
   the covariance, and add `reg_covar` to the diagonal.
6. **Initialization**: any reasonable scheme. The reference picks `K` distinct random rows of `X`
   as means using `torch.Generator().manual_seed(seed)`, gives every component the (biased)
   covariance of `X` plus `reg_covar · I`, and starts with equal weights. Don't consume the
   global RNG.
7. **`predict_proba(X)`** → `(M, K)` responsibilities. **`predict(X)`** → `(M,)` argmax.
   **`score(X)`** → `float`: the mean log-likelihood per sample.

---

### Constraints

- ✅ `torch.linalg` (`cholesky`, `solve_triangular`, `slogdet`, …), `torch.logsumexp`, broadcasting / `einsum`.
- ✅ Vectorize over samples and components. A Python loop over EM iterations is fine.
- ❌ Don't use `torch.distributions` in your implementation (the tests use it as the oracle).
- ❌ Don't compute densities with `exp` and normalize afterwards: $e^{-1000} = 0$ gives $0/0$.
- ❌ No scikit-learn.

---

<details>
  <summary>💡 Hint</summary>

  - $\log \mathcal N(x\mid\mu,\Sigma) = -\tfrac12\big[D\log 2\pi + \log\lvert\Sigma\rvert + (x-\mu)^{\top}\Sigma^{-1}(x-\mu)\big]$.
    Which pieces depend on $x$, and which only on the component?
  - With a Cholesky factor $\Sigma = LL^{\top}$: $\log\lvert\Sigma\rvert = 2\sum_i \log L_{ii}$, and the
    Mahalanobis term is $\lVert z\rVert^2$ where $Lz = x - \mu$, which is a triangular solve.
  - `X[None] - means[:, None]` has shape `(K, N, D)`. Batched `cholesky` / `solve_triangular`
    handle all components at once.
  - The weighted covariance of all components is a single `einsum`.
  - Before writing the loop, decide which parameters each history entry belongs to.

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
# Three 2D Gaussians with different shapes and unequal weights.
torch.manual_seed(0)
true_weights = torch.tensor([0.5, 0.3, 0.2])
true_means = torch.tensor([[0.0, 0.0], [5.0, 1.0], [1.0, 5.0]])
true_covs = torch.tensor([[[1.0, 0.6], [0.6, 1.0]],
                          [[0.5, -0.2], [-0.2, 0.8]],
                          [[0.3, 0.0], [0.0, 0.3]]])
counts = (true_weights * 600).long().tolist()                          # 300, 180, 120
X = torch.cat([m + torch.randn(n, 2) @ torch.linalg.cholesky(c).T      # x = mu + L eps
               for m, c, n in zip(true_means, true_covs, counts)])
X = X[torch.randperm(len(X))]
print("X", tuple(X.shape))

In [ ]:
class GaussianMixture:
    """Gaussian mixture with full covariances, fitted by expectation-maximization."""

    def __init__(self, n_components, max_iter=100, tol=1e-4, reg_covar=1e-6, seed=0):
        self.n_components = n_components
        self.max_iter = max_iter
        self.tol = tol
        self.reg_covar = reg_covar
        self.seed = seed

    def fit(self, X):
        """
        Args:
            X: (N, D) data.
        Returns:
            self, with weights_ (K,), means_ (K, D), covariances_ (K, D, D) and
            log_likelihood_history_ (list of floats; entry i = after i M-steps).
        """
        # TODO: initialize; record the mean log-likelihood; then alternate M-step / E-step,
        #       recording after each M-step and stopping when the gain drops below tol.
        raise NotImplementedError

    def predict_proba(self, X):
        """(M, D) -> (M, K) responsibilities p(component k | x)."""
        # TODO: log-space E-step
        raise NotImplementedError

    def predict(self, X):
        """(M, D) -> (M,) most likely component."""
        # TODO
        raise NotImplementedError

    def score(self, X):
        """Mean log-likelihood per sample, (1/M) sum_n log p(x_n), as a float."""
        # TODO
        raise NotImplementedError

In [ ]:
# Test 1: EM never decreases the likelihood; responsibilities are distributions, even far away
gmm = GaussianMixture(n_components=3, seed=0).fit(X)
h = gmm.log_likelihood_history_
print(f"{len(h) - 1} M-steps: mean log-likelihood {h[0]:.4f} -> {h[-1]:.4f}")
assert all(b >= a - 1e-6 for a, b in zip(h[:-1], h[1:])), "the log-likelihood decreased"
assert abs(h[-1] - gmm.score(X)) < 1e-4, "history[-1] must be the score of the final parameters"

R = gmm.predict_proba(X)
assert R.shape == (len(X), 3) and torch.allclose(R.sum(1), torch.ones(len(X)), atol=1e-5)
far = torch.tensor([[100.0, 100.0], [-500.0, 300.0]])
assert torch.isfinite(gmm.predict_proba(far)).all(), "NaN far from the data: do the E-step in log space"
assert torch.equal(gmm.predict(X), R.argmax(1))
print("Test 1 passed")

In [ ]:
# Test 2: score() against torch.distributions at the fitted parameters
gmm = GaussianMixture(n_components=3, seed=0).fit(X)
mvn = torch.distributions.MultivariateNormal(gmm.means_, covariance_matrix=gmm.covariances_)
log_joint = mvn.log_prob(X[:, None, :]) + gmm.weights_.log()            # (N, K)
reference = torch.logsumexp(log_joint, dim=1).mean().item()
print(f"score {gmm.score(X):.5f}  reference {reference:.5f}")
assert abs(gmm.score(X) - reference) < 1e-4
print("Test 2 passed")

In [ ]:
# Test 3: recover the generating parameters (best of 5 restarts, like sklearn's n_init)
best = max((GaussianMixture(3, seed=s).fit(X) for s in range(5)), key=lambda g: g.score(X))
match = torch.cdist(true_means, best.means_).argmin(dim=1)             # fitted component per true one
assert len(set(match.tolist())) == 3, "two true clusters were matched to the same component"
print("weights:", [round(v, 3) for v in best.weights_[match].tolist()], " true:", [round(v, 3) for v in true_weights.tolist()])
print("means  :", [[round(v, 2) for v in m] for m in best.means_[match].tolist()])
assert torch.allclose(best.weights_[match], true_weights, atol=0.03)
assert torch.allclose(best.means_[match], true_means, atol=0.2)
assert torch.allclose(best.covariances_[match], true_covs, atol=0.3)   # sampling noise, N = 600

# K = 1 is plain maximum likelihood: the sample mean and the biased sample covariance (+ reg)
one = GaussianMixture(n_components=1, reg_covar=1e-3).fit(X)
Xc = X - X.mean(0)
assert torch.allclose(one.means_[0], X.mean(0), atol=1e-4)
assert torch.allclose(one.covariances_[0], Xc.T @ Xc / len(X) + 1e-3 * torch.eye(2), atol=1e-4)
print("Test 3 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("gmm-em", GaussianMixture=GaussianMixture)